# DeepSpeed ZeRO-1 on 2×T4 (Kaggle)

Goal: **prove ZeRO-1 runs end-to-end** on DistilBERT + ag_news.
Then we upgrade → ZeRO-2 → ZeRO-3 → bigger model.

## ZeRO stages

| Stage | What is sharded | VRAM saved vs DDP |
|:--|:--|:--|
| **1 (today)** | Optimizer states (`m`,`v`) of AdamW | ~2× less |
| 2 (later)  | + gradients               | ~2× more |
| 3 (final)  | + parameters              | most  |

T4 ×2, no NVLink, NCCL over PCIe. fp16 required (no bf16 on sm_75).

In [1]:
!pip -q uninstall -y transformers huggingface_hub tokenizers || true
!pip -q install --no-deps \
  "transformers==4.57.0" \
  "huggingface_hub==0.35.3" \
  "tokenizers==0.22.1"
!pip -q install -U evaluate
!pip -q install deepspeed
!python -c "import deepspeed; print('deepspeed', deepspeed.__version__)"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.3 MB/s eta 0:00:00
Reason for being yanked: Error in the setup causing installation issues
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.8 MB/s eta 0:00:00
deepspeed 0.19.6


In [2]:
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
import multiprocessing as _mp
print("[OK] start_method =", _mp.get_start_method())


[OK] start_method = spawn


In [3]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:   ", torch.cuda.is_available())
print("GPUs:    ", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU[{i}]:", torch.cuda.get_device_name(i))
import deepspeed
print("DeepSpeed:", deepspeed.__version__)


PyTorch: 2.10.0+cu128
CUDA:    True
GPUs:     2
  GPU[0]: Tesla T4
  GPU[1]: Tesla T4


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


DeepSpeed: 0.19.6


In [4]:
%%bash
cat > ds_zero1.json << 'EOF'
{
  "fp16": {"enabled": true},
  "zero_optimization": {"stage": 1},
  "train_batch_size": 64,
  "train_micro_batch_size_per_gpu": 32,
  "gradient_accumulation_steps": 1
}
EOF
echo "[ok] ds_zero1.json"
cat ds_zero1.json

cat > train_ds.py << 'PY_END'
import os, time, json, random, argparse
import numpy as np
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TRANSFORMERS_NO_TORCHAO_IMPORT', '1')
os.environ.setdefault('NCCL_P2P_DISABLE', '1')
os.environ.setdefault('NCCL_IB_DISABLE', '1')
os.environ.setdefault('NCCL_ASYNC_ERROR_HANDLING', '1')
os.environ.setdefault("PYTHONUNBUFFERED", "1")
# ZeRO-1 relies on standard torch.optim.AdamW (no CUDA fused kernels, no nvcc at runtime).
# If DeepSpeed tries to compile ops we get `collect2: ld ... 1` warnings — they are
# non-fatal and the ops remain unused by ZeRO-1. No action needed.

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--out_dir", default="out_ds")
    p.add_argument("--steps",  type=int, default=300)
    p.add_argument("--per_device_train_bs", type=int, default=32)
    p.add_argument("--ds_config", default="ds_zero1.json")
    args, _ = p.parse_known_args()

    from datasets import load_dataset
    from transformers import (
        AutoTokenizer, DataCollatorWithPadding,
        AutoModelForSequenceClassification,
        TrainingArguments, Trainer, EvalPrediction
    )
    import torch, deepspeed
    SEED = 42
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    print(f'[rank {os.environ.get("LOCAL_RANK","?")} / world {os.environ.get("WORLD_SIZE","?")}]')

    ds = load_dataset("ag_news")
    tr = ds["train"].shuffle(seed=SEED).select(range(6000))
    te = ds["test"].shuffle(seed=SEED).select(range(2000))
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)
    def tok_fn(b): return tok(b["text"], truncation=True, padding=False, max_length=128)
    tr_tok = tr.map(tok_fn, batched=True, remove_columns=["text"])
    te_tok = te.map(tok_fn, batched=True, remove_columns=["text"])
    coll = DataCollatorWithPadding(tok, pad_to_multiple_of=8)
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4)
    model.gradient_checkpointing_enable()

    def compute_metrics(ep: EvalPrediction):
        return {"accuracy": float((ep.predictions.argmax(-1) == ep.label_ids).mean())}

    args_tr = TrainingArguments(
        output_dir=args.out_dir,
        per_device_train_batch_size=args.per_device_train_bs,
        per_device_eval_batch_size=128,
        learning_rate=2e-5,
        fp16=True,
        max_steps=args.steps,
        eval_strategy="no",
        save_strategy="no",
        dataloader_num_workers=2,
        dataloader_persistent_workers=False,
        dataloader_pin_memory=True,
        report_to="none",
        logging_steps=50,
        deepspeed=args.ds_config,
    )

    t0 = time.perf_counter()
    trainer = Trainer(model=model, args=args_tr,
                     train_dataset=tr_tok, eval_dataset=te_tok,
                     data_collator=coll, tokenizer=tok,
                     compute_metrics=compute_metrics)
    print(f'[check] deepspeed_enabled={trainer.is_deepspeed_enabled}  ' + f'v={deepspeed.__version__}')
    trainer.train()
    # Direct runtime proof: engine + optimizer classes are DeepSpeed's own
    wrapped = getattr(trainer, 'model_wrapped', None)
    eng_name = type(wrapped).__name__ if wrapped else 'n/a'
    opt_name = type(trainer.optimizer).__name__
    try:
        import deepspeed as _ds
        is_ds_eng = wrapped is not None and isinstance(wrapped, _ds.DeepSpeedEngine)
        is_ddp    = wrapped is not None and 'DistributedDataParallel' in eng_name
        print(f'[proof] engine={eng_name}  is_DeepSpeedEngine={is_ds_eng}  is_DDP={is_ddp}')
        print(f'[proof] optimizer={opt_name}')
    except Exception as e:
        print(f'[proof] (probe failed: {e})')
    dt = time.perf_counter() - t0
    ev   = trainer.evaluate()
    res = {
        "world_size":  os.environ.get("WORLD_SIZE","1"),
        "zero_stage":  1,
        "engine_cls":  eng_name,
        "optimizer_cls": type(trainer.optimizer).__name__,
        "steps/sec":   round(args.steps/dt, 2) if dt > 0 else 0,
        "final_acc":   round(ev.get("eval_accuracy", 0.0), 4),
    }
    os.makedirs(args.out_dir, exist_ok=True)
    with open(os.path.join(args.out_dir, "metrics.json"), "w") as f:
        json.dump(res, f, indent=2)
    print(res)

if __name__ == "__main__":
    main()
PY_END
echo "[ok] train_ds.py"


[ok] ds_zero1.json
{
  "fp16": {"enabled": true},
  "zero_optimization": {"stage": 1},
  "train_batch_size": 64,
  "train_micro_batch_size_per_gpu": 32,
  "gradient_accumulation_steps": 1
}
[ok] train_ds.py


In [5]:
import os, subprocess, shlex, json

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"]       = "false"
env["TRANSFORMERS_NO_TORCHAO_IMPORT"] = "1"
env["NCCL_P2P_DISABLE"]             = "1"
env["NCCL_IB_DISABLE"]              = "1"
env["NCCL_ASYNC_ERROR_HANDLING"]    = "1"
env["PYTHONUNBUFFERED"]             = "1"

# NOTE: launch with `accelerate` (not the deepspeed CLI).
# The deepspeed CLI sets CUDA_VISIBLE_DEVICES globally and can break
# rank-1 device mapping on Kaggle. DeepSpeed is wired via
# TrainingArguments(deepspeed=ds_zero1.json) instead.

CMD = [
    "accelerate", "launch",
    "--num_processes", "2",
    "--mixed_precision", "fp16",
    "train_ds.py",
    "--out_dir", "out_ds",
    "--per_device_train_bs", "32",
    "--steps", "300",
    "--ds_config", "ds_zero1.json",
]
print(">>", " ".join(CMD), flush=True)

proc = subprocess.Popen(
    CMD, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
print(f"\n[exit code]: {ret}")

try:
    print("[result]", json.load(open("out_ds/metrics.json")))
except Exception as e:
    print("[no metrics]", e)


>> accelerate launch --num_processes 2 --mixed_precision fp16 train_ds.py --out_dir out_ds --per_device_train_bs 32 --steps 300 --ds_config ds_zero1.json
The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[rank 0 / world 2]
[rank 1 / world 2]

Generating train split: 100%|██████████| 120000/120000 [00:00<00:00, 649770.95 examples/s]

Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 675525.78 examples/s]

Map: 100%|██████████| 6000/6000 [00:01<00:00, 4545.33 examples/s]

Map: 100%|██████████| 6000/6000 [00:01<00:00, 4480.07 examples/s]

Map: 100%|██████████| 2000/2000 [00:00<00:00, 4613.50 examples/s]

Map: 100%|████████

## How to confirm ZeRO-1 is active

| Log line | Meaning |
|:--|:--|
| `deepspeed_enabled=True` | Trainer uses DeepSpeed, not DDP |
| `zero_optimization: stage 1` | ZeRO-1 config loaded |
| `world_size=2` | Both T4 joined |
| `final_acc` in `output_ds/metrics.json` | Training finished |

## Upgrade path

1. ZeRO-2 → change `"stage": 1` → `"stage": 2` in `ds_zero1.json`
2. ZeRO-3 → change to `"stage": 3`
3. Bigger model → swap `distilbert-base-uncased` → `bert-base-uncased`
4. Full data → remove `.select(range(6000))`
